 # DAG + refutation tests



 A CATE number by itself is just a number from a library. This notebook

 is about making the causal assumptions explicit and then actually

 stress-testing them, which is the part I think is easiest to skip and

 most important not to.

In [1]:
import numpy as np
import pandas as pd
from causaldata import nhefs
from dowhy import CausalModel
from sklearn.impute import SimpleImputer
%matplotlib inline

RNG = 42
np.random.seed(RNG)

CONFOUNDERS = [
    "age", "sex", "race", "education", "smokeintensity", "smokeyrs",
    "exercise", "active", "wt71", "ht",
]
TREATMENT = "qsmk"
OUTCOME = "wt82_71"


 ### The DAG



 Every confounder points into both treatment and outcome (that's what

 makes it a confounder), and treatment points into outcome. I kept this

 deliberately simple — no mediators, no colliders — mostly because I

 wanted something I could clearly defend rather than something more

 elaborate I'd be less sure about. That's a real limitation, not just a

 caveat I'm adding for form's sake.

In [2]:
GRAPH = "digraph {"
for c in CONFOUNDERS:
    GRAPH += f'"{c}" -> "{TREATMENT}"; "{c}" -> "{OUTCOME}"; '
GRAPH += f'"{TREATMENT}" -> "{OUTCOME}";'
GRAPH += "}"


In [3]:
def load_data():
    df = nhefs.load_pandas().data
    cols = CONFOUNDERS + [TREATMENT, OUTCOME]
    df = df[cols].dropna(subset=[TREATMENT, OUTCOME]).copy()
    imputer = SimpleImputer(strategy="median")
    df[CONFOUNDERS] = imputer.fit_transform(df[CONFOUNDERS])
    return df

df = load_data()


 ### Identification



 `identify_effect` runs the backdoor criterion against the graph I gave

 it — it works out which variables actually need adjusting for, rather

 than me just guessing.

In [4]:
model = CausalModel(data=df, treatment=TREATMENT, outcome=OUTCOME, graph=GRAPH)
identified_estimand = model.identify_effect(proceed_when_unidentifiable=True)
print(identified_estimand)


Estimand type: EstimandType.NONPARAMETRIC_ATE

### Estimand : 1
Estimand name: backdoor
Estimand expression:
   d                                                                           ↪
───────(E[wt_82_71|age,race,wt71,smokeintensity,ht,smokeyrs,exercise,sex,educa ↪
d[qsmk]                                                                        ↪

↪              
↪ tion,active])
↪              
Estimand assumption 1, Unconfoundedness: If U→{qsmk} and U→wt82_71 then P(wt82_71|qsmk,age,race,wt71,smokeintensity,ht,smokeyrs,exercise,sex,education,active,U) = P(wt82_71|qsmk,age,race,wt71,smokeintensity,ht,smokeyrs,exercise,sex,education,active)

### Estimand : 2
Estimand name: iv
No such variable(s) found!

### Estimand : 3
Estimand name: frontdoor
No such variable(s) found!

### Estimand : 4
Estimand name: general_adjustment
Estimand expression:
   d                                                                           ↪
───────(E[wt_82_71|age,race,wt71,smokeintensity,ht,smokeyrs,ex

In [5]:
estimate = model.estimate_effect(identified_estimand, method_name="backdoor.linear_regression")
print(f"ATE = {estimate.value:+.2f} kg")


ATE = +3.34 kg


 ### Refutation 1 — placebo treatment



 Shuffle the treatment column into noise and re-run. If there's really

 an effect, a fake randomized treatment shouldn't show one.

In [6]:
refute_placebo = model.refute_estimate(
    identified_estimand, estimate,
    method_name="placebo_treatment_refuter",
    placebo_type="permute",
    random_state=RNG,
)
print(refute_placebo)


Refute: Use a Placebo Treatment
Estimated effect:3.3373235832882386
New effect:-0.03481017724970932
p value:0.88



 ### Refutation 2 — random common cause



 Add a meaningless random variable to the confounder set. A stable

 estimate shouldn't move much when you do this.

In [7]:
refute_random_cause = model.refute_estimate(
    identified_estimand, estimate,
    method_name="random_common_cause",
    random_state=RNG,
)
print(refute_random_cause)


Refute: Add a random common cause
Estimated effect:3.3373235832882386
New effect:3.338321573236131
p value:0.9199999999999999



 Both passed — placebo collapsed to basically zero (p=0.88), and the

 random common cause barely moved the estimate at all. That doesn't

 prove the DAG is correct, but it's evidence the result isn't just an

 artifact of how the model happens to be specified, which is about as

 much confidence as you can reasonably claim from checks like these.